# Phase 3: acquire and verify the source data

This phase downloads and checks the original StatsBomb files for La Liga 2015/2016. It does not yet select shots, create labels, or train a model.

The downloader pins repository revision 4b73468fc5b0f1950f9f66fada70ad3a4f9327cb, competition ID 11, and season ID 27, with an expected 380-match schedule. A pinned revision keeps every download URL on the same saved repository version.

The data are JSON files. The match index lists the games; each game has a separate event file. An event might be a pass, foul, or shot, so event count is not shot count. Original files are preserved in data/raw/statsbomb/<commit>/. The manifest records each file's source URL, byte count, retrieval time, and SHA-256 fingerprint. Matching the saved fingerprint detects later changes to a local copy.

The downloader uses only Python's standard library. Its first run needs internet access; later runs reuse only files verified against the manifest. Keep the upstream README and licence with the data and credit StatsBomb when sharing analysis.


In [ ]:
from pathlib import Path
import sys

starting_directory = Path.cwd().resolve()
search_locations = (starting_directory, *starting_directory.parents)
project_root = next(
    (
        folder
        for folder in search_locations
        if (folder / ".venv").is_dir()
        and (folder / "README.md").is_file()
        and (folder / "scripts" / "acquire_data.py").is_file()
    ),
    None,
)
assert project_root is not None, "Open the project folder in VS Code before running this notebook."
script_path = project_root / "scripts" / "acquire_data.py"
raw_root = (
    project_root
    / "data"
    / "raw"
    / "statsbomb"
    / "4b73468fc5b0f1950f9f66fada70ad3a4f9327cb"
)
manifest_path = raw_root / "manifest.json"

print(f"Notebook started in: {starting_directory}")
print(f"Project folder:      {project_root}")
print(f"Python interpreter:  {sys.executable}")
print(f"Downloader script:   {script_path}")


## 1. Acquire missing files

The first run may take time because this season has 380 separate event files. Progress is shown every 20 files. The manifest is saved as files are checked, so an interrupted download can be resumed. The downloader does not delete raw files or silently overwrite a file whose saved hash no longer matches.


In [ ]:
import subprocess

acquisition = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=project_root,
    check=True,
    text=True,
)


## 2. Read the acquisition inventory

The manifest lists files and event counts. These counts do not tell us how many shots there are; shot extraction is a later phase.


In [ ]:
import json

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
file_records = manifest["files"]
event_records = [
    record
    for file_path, record in file_records.items()
    if file_path.startswith("data/events/") and file_path.endswith(".json")
]

print(f"Snapshot complete: {manifest['complete']}")
print(f"Repository commit:  {manifest['commit']}")
print(f"Competition/season: {manifest['competition_id']}/{manifest['season_id']}")
print(f"Expected matches:   {manifest['expected_match_count']}")
print(f"Event files listed: {len(event_records)}")
print(f"Individual events:  {sum(record['event_count'] for record in event_records)}")
print(f"Files listed:       {len(file_records)}")
print(f"Manifest:           {manifest_path}")
assert manifest["complete"] is True
assert len(event_records) == manifest["expected_match_count"]


## 3. Inspect a small raw event sample

This preview reads an original event JSON file and displays a few event types. It checks the source format; it does not transform events into model rows.


In [ ]:
match_index_path = raw_root / "data" / "matches" / "11" / "27.json"
matches = json.loads(match_index_path.read_text(encoding="utf-8"))
first_match_id = matches[0]["match_id"]
sample_event_path = raw_root / "data" / "events" / f"{first_match_id}.json"
sample_events = json.loads(sample_event_path.read_text(encoding="utf-8"))

print(f"Sample match ID: {first_match_id}")
print(f"Events in this file: {len(sample_events)}")
for event in sample_events[:5]:
    event_type = event.get("type", {})
    team = event.get("team", {})
    print({
        "id": event.get("id"),
        "type": event_type.get("name") if isinstance(event_type, dict) else None,
        "minute": event.get("minute"),
        "team": team.get("name") if isinstance(team, dict) else None,
    })


## 4. Verify offline

Verification reads local files and their manifest. It makes no network requests. A successful result confirms the pinned files are present and still match their saved SHA-256 hashes and structural checks.


In [ ]:
verification = subprocess.run(
    [sys.executable, str(script_path), "--verify-only"],
    cwd=project_root,
    check=True,
    text=True,
)
